# Lesson 15 — GroupBy & Aggregation in Pandas

## Quality Engineering / Manufacturing Data Analysis

**Goal:** Learn how to group manufacturing data and calculate meaningful quality KPIs using `groupby()`, aggregation, `agg()`, `transform()`, filtering, and multi-level grouping.

### What you will learn
- `groupby()` fundamentals
- Single and multiple-column grouping
- `sum()`, `mean()`, `count()`, `min()`, `max()`, `std()`
- Named aggregation with `agg()`
- Group-level filtering
- `transform()` for row-level comparison with group KPIs
- Defect and yield analysis
- Practical manufacturing QA examples
- Common mistakes and QA-oriented interpretation


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')


## 1. Create a Manufacturing Quality Dataset

We will work with inspection records from an electronics manufacturing line.

In [2]:
df = pd.DataFrame({
    'Date': pd.to_datetime([
        '2026-09-01','2026-09-01','2026-09-01','2026-09-01','2026-09-02',
        '2026-09-02','2026-09-02','2026-09-02','2026-09-03','2026-09-03',
        '2026-09-03','2026-09-03','2026-09-04','2026-09-04','2026-09-04',
        '2026-09-04','2026-09-05','2026-09-05','2026-09-05','2026-09-05'
    ]),
    'Line': ['L1','L1','L2','L2','L1','L1','L2','L2','L1','L1','L2','L2','L1','L1','L2','L2','L1','L1','L2','L2'],
    'Shift': ['A','B','A','B','A','B','A','B','A','B','A','B','A','B','A','B','A','B','A','B'],
    'Model': ['M1','M1','M1','M2','M1','M2','M1','M2','M1','M2','M1','M2','M1','M2','M1','M2','M1','M2','M1','M2'],
    'Inspected': [980,950,1020,990,1000,970,1010,980,1030,1000,1020,995,990,960,1040,1000,1015,985,1035,1010],
    'Defects': [18,25,12,30,15,28,20,24,17,31,14,26,22,29,13,27,16,24,19,32],
    'Cycle_Time_sec': [42,44,41,46,43,45,42,47,41,46,43,45,42,47,41,46,43,44,42,48]
})

df['Good'] = df['Inspected'] - df['Defects']
df['Defect_Rate'] = df['Defects'] / df['Inspected'] * 100
df['Yield'] = df['Good'] / df['Inspected'] * 100

df.head()

,Date,Line,Shift,Model,Inspected,Defects,Cycle_Time_sec,Good,Defect_Rate,Yield
0,2026-09-01,L1,A,M1,980,18,42,962,1.837,98.163
1,2026-09-01,L1,B,M1,950,25,44,925,2.632,97.368
2,2026-09-01,L2,A,M1,1020,12,41,1008,1.176,98.824
3,2026-09-01,L2,B,M2,990,30,46,960,3.030,96.970
4,2026-09-02,L1,A,M1,1000,15,43,985,1.500,98.500


## 2. Why `groupby()` matters in Quality

A QA engineer rarely wants only row-level information. Typical questions are:

- Which production line has the highest defect rate?
- Which shift has lower yield?
- Which model creates the most defects?
- What is the average cycle time by line and shift?
- Which line-model combination needs investigation?

`groupby()` converts these questions into repeatable calculations.

In [3]:
# Basic syntax
# df.groupby('Column')['NumericColumn'].mean()

line_defects = df.groupby('Line')['Defects'].sum()
line_defects

Line
L1    225
L2    217
Name: Defects, dtype: int64

## 3. Single-column grouping

In [4]:
df.groupby('Line')['Inspected'].sum()

Line
L1     9880
L2    10100
Name: Inspected, dtype: int64

In [5]:
df.groupby('Line')['Defect_Rate'].mean()

Line
L1   2.286
L2   2.158
Name: Defect_Rate, dtype: float64

### Important QA point

The **mean of row-level defect rates** is not always the correct overall defect rate.

For quality reporting, a weighted rate is usually calculated as:

`Total Defects / Total Inspected × 100`

In [6]:
line_kpi = df.groupby('Line').agg(
    Total_Inspected=('Inspected', 'sum'),
    Total_Defects=('Defects', 'sum')
)

line_kpi['Defect_Rate'] = line_kpi['Total_Defects'] / line_kpi['Total_Inspected'] * 100
line_kpi['Yield'] = 100 - line_kpi['Defect_Rate']
line_kpi

,Total_Inspected,Total_Defects,Defect_Rate,Yield
Line,,,,
L1,9880,225,2.277,97.723
L2,10100,217,2.149,97.851


## 4. Multiple-column grouping

In [7]:
df.groupby(['Line', 'Shift'])['Defects'].sum()

Line  Shift
L1    A         88
      B        137
L2    A         78
      B        139
Name: Defects, dtype: int64

In [8]:
df.groupby(['Line', 'Shift'])[['Inspected', 'Defects', 'Good']].sum()

Inspected  Defects  Good
Line Shift                          
L1   A           5015       88  4927
     B           4865      137  4728
L2   A           5125       78  5047
     B           4975      139  4836

## 5. `agg()` — the most useful QA pattern

With `agg()` we can calculate several statistics at the same time.

In [9]:
summary = df.groupby('Line').agg({
    'Inspected': ['sum', 'mean'],
    'Defects': ['sum', 'mean', 'max'],
    'Cycle_Time_sec': ['mean', 'min', 'max', 'std']
})

summary

Inspected          Defects            Cycle_Time_sec              
           sum     mean     sum   mean max           mean min max   std
Line                                                                   
L1        9880  988.000     225 22.500  31         43.700  41  47 1.889
L2       10100 1010.000     217 21.700  32         44.100  41  48 2.601

## 6. Named aggregation — recommended style

Named aggregation produces clean column names and is easier to use in dashboards.

In [10]:
qa_summary = df.groupby('Line').agg(
    Total_Inspected=('Inspected', 'sum'),
    Total_Defects=('Defects', 'sum'),
    Avg_Cycle_Time=('Cycle_Time_sec', 'mean'),
    Max_Cycle_Time=('Cycle_Time_sec', 'max'),
    Cycle_Time_SD=('Cycle_Time_sec', 'std')
)

qa_summary['Defect_Rate'] = qa_summary['Total_Defects'] / qa_summary['Total_Inspected'] * 100
qa_summary['Yield'] = 100 - qa_summary['Defect_Rate']
qa_summary.round(2)

,Total_Inspected,Total_Defects,Avg_Cycle_Time,Max_Cycle_Time,Cycle_Time_SD,Defect_Rate,Yield
Line,,,,,,,
L1,9880,225,43.700,47,1.890,2.280,97.720
L2,10100,217,44.100,48,2.600,2.150,97.850


## 7. Group by Line + Shift

This is useful when the same line behaves differently across shifts.

In [11]:
line_shift = df.groupby(['Line', 'Shift']).agg(
    Inspected=('Inspected', 'sum'),
    Defects=('Defects', 'sum'),
    Avg_Cycle_Time=('Cycle_Time_sec', 'mean')
)

line_shift['Defect_Rate'] = line_shift['Defects'] / line_shift['Inspected'] * 100
line_shift['Yield'] = 100 - line_shift['Defect_Rate']
line_shift.round(2)

Inspected  Defects  Avg_Cycle_Time  Defect_Rate  Yield
Line Shift                                                        
L1   A           5015       88          42.200        1.750 98.250
     B           4865      137          45.200        2.820 97.180
L2   A           5125       78          41.800        1.520 98.480
     B           4975      139          46.400        2.790 97.210

## 8. `reset_index()`

After grouping, the grouping columns may become an index. Use `reset_index()` when you want a normal DataFrame for further analysis or export.

In [12]:
line_shift_df = line_shift.reset_index()
line_shift_df

,Line,Shift,Inspected,Defects,Avg_Cycle_Time,Defect_Rate,Yield
0,L1,A,5015,88,42.200,1.755,98.245
1,L1,B,4865,137,45.200,2.816,97.184
2,L2,A,5125,78,41.800,1.522,98.478
3,L2,B,4975,139,46.400,2.794,97.206


## 9. `transform()` — keep group KPI beside every row

`agg()` reduces the number of rows. `transform()` keeps the original number of rows and broadcasts a group-level result back to each record.

This is extremely useful for outlier detection and comparison against a line/shift baseline.

In [13]:
df['Line_Avg_Cycle'] = df.groupby('Line')['Cycle_Time_sec'].transform('mean')
df['Cycle_vs_Line_Avg'] = df['Cycle_Time_sec'] - df['Line_Avg_Cycle']

df[['Line', 'Shift', 'Cycle_Time_sec', 'Line_Avg_Cycle', 'Cycle_vs_Line_Avg']].head(10)

,Line,Shift,Cycle_Time_sec,Line_Avg_Cycle,Cycle_vs_Line_Avg
0,L1,A,42,43.700,-1.700
1,L1,B,44,43.700,0.300
2,L2,A,41,44.100,-3.100
3,L2,B,46,44.100,1.900
4,L1,A,43,43.700,-0.700
5,L1,B,45,43.700,1.300
6,L2,A,42,44.100,-2.100
7,L2,B,47,44.100,2.900
8,L1,A,41,43.700,-2.700
9,L1,B,46,43.700,2.300


## 10. Find records above the group average

In [14]:
df[df['Cycle_Time_sec'] > df['Line_Avg_Cycle']]

,Date,Line,Shift,Model,Inspected,Defects,Cycle_Time_sec,Good,Defect_Rate,Yield,Line_Avg_Cycle,Cycle_vs_Line_Avg
1,2026-09-01,L1,B,M1,950,25,44,925,2.632,97.368,43.700,0.300
3,2026-09-01,L2,B,M2,990,30,46,960,3.030,96.970,44.100,1.900
5,2026-09-02,L1,B,M2,970,28,45,942,2.887,97.113,43.700,1.300
7,2026-09-02,L2,B,M2,980,24,47,956,2.449,97.551,44.100,2.900
9,2026-09-03,L1,B,M2,1000,31,46,969,3.100,96.900,43.700,2.300
11,2026-09-03,L2,B,M2,995,26,45,969,2.613,97.387,44.100,0.900
13,2026-09-04,L1,B,M2,960,29,47,931,3.021,96.979,43.700,3.300
15,2026-09-04,L2,B,M2,1000,27,46,973,2.700,97.300,44.100,1.900
17,2026-09-05,L1,B,M2,985,24,44,961,2.437,97.563,43.700,0.300
19,2026-09-05,L2,B,M2,1010,32,48,978,3.168,96.832,44.100,3.900


## 11. Group-level filtering with `filter()`

Suppose we only want lines whose total inspected quantity is above 9,500 units.

In [15]:
high_volume_lines = df.groupby('Line').filter(
    lambda g: g['Inspected'].sum() > 9500
)

high_volume_lines['Line'].unique()

array(['L1', 'L2'], dtype=object)

## 12. Model-wise quality analysis

In [16]:
model_kpi = df.groupby('Model').agg(
    Total_Inspected=('Inspected', 'sum'),
    Total_Defects=('Defects', 'sum'),
    Avg_Cycle_Time=('Cycle_Time_sec', 'mean')
)

model_kpi['Defect_Rate'] = model_kpi['Total_Defects'] / model_kpi['Total_Inspected'] * 100
model_kpi['Yield'] = 100 - model_kpi['Defect_Rate']
model_kpi.round(2)

,Total_Inspected,Total_Defects,Avg_Cycle_Time,Defect_Rate,Yield
Model,,,,,
M1,11090,191,42.180,1.720,98.280
M2,8890,251,46.000,2.820,97.180


## 13. Date-wise quality trend

Grouping by date is the foundation for daily quality reporting.

In [17]:
daily = df.groupby('Date').agg(
    Inspected=('Inspected', 'sum'),
    Defects=('Defects', 'sum')
)

daily['Defect_Rate'] = daily['Defects'] / daily['Inspected'] * 100
daily['Yield'] = 100 - daily['Defect_Rate']
daily.round(2)

,Inspected,Defects,Defect_Rate,Yield
Date,,,,
2026-09-01,3940,85,2.160,97.840
2026-09-02,3960,87,2.200,97.800
2026-09-03,4045,88,2.180,97.820
2026-09-04,3990,91,2.280,97.720
2026-09-05,4045,91,2.250,97.750


## 14. Top problem groups

For QA prioritization, sort the grouped result by defect rate.

In [18]:
problem_groups = df.groupby(['Line', 'Shift']).agg(
    Inspected=('Inspected', 'sum'),
    Defects=('Defects', 'sum')
)

problem_groups['Defect_Rate'] = problem_groups['Defects'] / problem_groups['Inspected'] * 100
problem_groups.sort_values('Defect_Rate', ascending=False).round(2)

,,Inspected,Defects,Defect_Rate
Line,Shift,,,
L1,B,4865,137,2.820
L2,B,4975,139,2.790
L1,A,5015,88,1.750
L2,A,5125,78,1.520


## 15. QA Exercise — do it yourself

Try these without looking at the solution first:

1. Calculate total inspected quantity by Shift.
2. Calculate total defects by Model.
3. Calculate weighted defect rate by Model.
4. Calculate average cycle time by Line and Model.
5. Find the Line + Shift combination with the highest defect rate.
6. Add a `Model_Avg_Cycle` column using `transform()`.
7. Identify records where cycle time is more than 2 seconds above the model average.


In [19]:
# Exercise 1
# df.groupby('Shift')['Inspected'].sum()

# Exercise 2
# df.groupby('Model')['Defects'].sum()

# Exercise 3
# model = df.groupby('Model').agg(Inspected=('Inspected','sum'), Defects=('Defects','sum'))
# model['Defect_Rate'] = model['Defects'] / model['Inspected'] * 100

# Exercise 4
# df.groupby(['Line','Model'])['Cycle_Time_sec'].mean()

# Exercise 5
# Use the same weighted defect-rate logic shown earlier.

# Exercise 6
# df['Model_Avg_Cycle'] = df.groupby('Model')['Cycle_Time_sec'].transform('mean')

# Exercise 7
# df[df['Cycle_Time_sec'] > df['Model_Avg_Cycle'] + 2]


# Key Takeaways

### Remember these 7 patterns

1. `groupby()` → divide data into meaningful groups.
2. `sum()` → total production/defects.
3. `mean()` → average performance.
4. `agg()` → multiple KPIs in one operation.
5. Weighted rate → `total defects / total inspected` for overall defect rate.
6. `transform()` → compare every row against its group benchmark.
7. `reset_index()` → convert grouped indexes back into normal columns.

### QA mindset

**Don't just calculate an average. Ask: average of what, at which level, and with what denominator?**

That distinction becomes critical when you build quality dashboards, Pareto analysis, control charts, and management reports.


# Mini Project — Production Quality Dashboard Dataset

Use `df` to create a final KPI table with:

- Line
- Shift
- Model
- Total Inspected
- Total Defects
- Defect Rate %
- Yield %
- Average Cycle Time
- Cycle Time Standard Deviation

Then sort it by Defect Rate descending and identify which combinations deserve further root-cause investigation.